# Phase 2 — v3 Recipe Validation: cls_pw + capped multi-scale + freeze warm-start

The two-model class-routed ensemble (notebook 06) was rejected in favor of a single model that
generalises across all 4 classes on its own. This notebook validates three recipe changes on top
of the v2 baseline (`copy_paste=0.6`, `mixup=0.15`, `cos_lr=True`, `lrf=0.001`) using a fast,
low-VRAM proxy: **YOLO26n**, 3-fold stratified cross-validation, ~2hr total budget.

**New in this recipe:**
1. `cls_pw=0.35` — native ultralytics inverse-class-frequency loss weighting (dampened, not full
   1.0 — Actinomyces already overfit once at 42 instances; full inverse-frequency would weight its
   loss ~23x relative to BV and risks reigniting that overfit).
2. `multi_scale=0.25` (was 0.5 in v2) — caps the downscale floor so TV's ~32×37px boxes don't train
   at sub-detectable resolutions (the likely cause of TV's small regression in v2).
3. `freeze=10` + an `on_train_epoch_start` callback that restores `requires_grad` on the backbone
   at epoch 10 — a warm-start that protects pretrained low-level features from being overwritten in
   early epochs on this small a dataset. No optimizer rebuild is needed for the unfreeze: ultralytics'
   `build_optimizer` groups *all* `named_parameters()` regardless of `requires_grad`
   (`engine/trainer.py:1023-1034`), so the frozen params are already in the optimizer's param groups —
   flipping `requires_grad` back on is sufficient.

**Validation set-up:** pool `train/` + `val/` (425 images) and stratify into 3 folds by rare-class
presence (Actinomyces has only ~20 carrying images total — a naive random split risks a fold with
near-zero Actinomyces). `test/` (48 images) stays completely untouched throughout, same principle
established in notebook 06.

**Scope note:** this validates the *recipe* using YOLO26n as a fast/cheap proxy — it is not the
final deployment model. If the recipe looks stable here, training the actual YOLO26s deployment
checkpoint on the full pooled data is a separate follow-up session.

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import yaml

ROOT           = Path('../../../').resolve()
DATA_ROOT      = ROOT / 'data/organisms_data'
CONFIG_DIR     = ROOT / 'models/organism_det/configs'
KFOLD_DIR      = CONFIG_DIR / 'kfold'
METRICS_DIR    = ROOT / 'models/organism_det/results/metrics'
CHECKPOINT_DIR = ROOT / 'models/organism_det/checkpoints' / 'kfold_v3'

for d in [KFOLD_DIR, METRICS_DIR, CHECKPOINT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = [
    'Trichomonas vaginalis',
    'Bacterial vaginosis flora shift',
    'Candida spp.',
    'Actinomyces spp.',
]
NC    = len(CLASS_NAMES)
K     = 3
IMGSZ = 1280

# v3 recipe under validation (proxy model: yolo26n, fast + low VRAM)
RECIPE = dict(
    epochs=100,
    patience=20,
    batch=8,
    imgsz=IMGSZ,
    workers=4,
    cls_pw=0.35,        # NEW -- native inverse-frequency class-loss weighting (dampened)
    multi_scale=0.25,   # v2 used 0.5 (640-1952px) -- likely shrank TV below its detection floor
    freeze=10,          # NEW -- freeze backbone for first 10 epochs, restored by callback below
    copy_paste=0.6,     # validated in v2
    mixup=0.15,         # validated in v2
    cos_lr=True,        # validated in v2
    lrf=0.001,          # validated in v2
    flipud=0.5,
    fliplr=0.5,
    degrees=15.0,
    hsv_h=0.05,
    hsv_s=0.3,
    hsv_v=0.1,
    scale=0.5,
)

assert (DATA_ROOT / 'test/images').exists(), 'test split missing -- it must stay untouched throughout'

### Pool train+val, stratify into K folds by rare-class presence

No physical file copying: each fold's `train`/`val` value in its dataset YAML is a `.txt` file of
absolute image paths, which ultralytics reads natively as a flat list (`data/base.py:get_img_files`)
regardless of which physical directory the image originally lived in — label paths are still
auto-derived by swapping `images/`→`labels/`.

In [3]:
def parse_label_counts(label_path: Path) -> np.ndarray:
    counts = np.zeros(NC, dtype=int)
    if not label_path.exists():
        return counts
    for line in label_path.read_text().strip().splitlines():
        if not line.strip():
            continue
        counts[int(line.split()[0])] += 1
    return counts


# Pool train + val (425 images) -- test/ (48 images) stays untouched as the one honest final check
pooled = []
for split in ['train', 'val']:
    img_dir = DATA_ROOT / split / 'images'
    label_dir = DATA_ROOT / split / 'labels'
    for img_path in sorted(img_dir.glob('*.jpg')):
        counts = parse_label_counts(label_dir / f'{img_path.stem}.txt')
        pooled.append({'path': str(img_path.resolve()), 'counts': counts})

print(f'Pooled {len(pooled)} images from train+val for {K}-fold split.')

# Sort rare-class-carrying images first: Actinomyces > TV > Candida > busiest images
pooled.sort(
    key=lambda im: (im['counts'][3] > 0, im['counts'][0] > 0, im['counts'][2] > 0, im['counts'].sum()),
    reverse=True,
)

# Greedy load-balanced assignment: weight rarest classes heaviest so no fold starves on Actinomyces/TV
RARITY_WEIGHTS = np.array([10, 1, 3, 100])  # TV, BV, Candida, Actinomyces
fold_images = [[] for _ in range(K)]
fold_load = np.zeros(K)
fold_size = np.zeros(K, dtype=int)

for im in pooled:
    score = float((im['counts'] * RARITY_WEIGHTS).sum())
    idx = int(np.lexsort((fold_size, fold_load))[0])  # lowest load first, ties -> fewest images
    fold_images[idx].append(im)
    fold_load[idx] += score
    fold_size[idx] += 1

# Sanity check BEFORE spending any training time: print per-fold class distribution
dist_rows = []
for i, imgs in enumerate(fold_images):
    fold_counts = np.sum([im['counts'] for im in imgs], axis=0)
    dist_rows.append({'fold': i, 'n_images': len(imgs), **{CLASS_NAMES[c]: int(fold_counts[c]) for c in range(NC)}})
dist_df = pd.DataFrame(dist_rows)
print(dist_df.to_string(index=False))
assert (dist_df[CLASS_NAMES[3]] > 0).all(), 'A fold has zero Actinomyces instances -- stratification failed.'

Pooled 425 images from train+val for 3-fold split.
 fold  n_images  Trichomonas vaginalis  Bacterial vaginosis flora shift  Candida spp.  Actinomyces spp.
    0       142                    110                              305            75                12
    1       142                    111                              298            74                12
    2       141                    111                              298            74                12


In [4]:
fold_yaml_paths = []
for i in range(K):
    val_imgs = fold_images[i]
    train_imgs = [im for j in range(K) if j != i for im in fold_images[j]]

    train_txt = KFOLD_DIR / f'fold{i}_train.txt'
    val_txt = KFOLD_DIR / f'fold{i}_val.txt'
    train_txt.write_text('\n'.join(im['path'] for im in train_imgs))
    val_txt.write_text('\n'.join(im['path'] for im in val_imgs))

    # 'train'/'val' as absolute paths resolve as-is regardless of 'path:' (pathlib join with an
    # absolute path discards the base) -- see ultralytics/data/utils.py check_det_dataset:512
    fold_yaml = {
        'path': str(DATA_ROOT),
        'train': str(train_txt.resolve()),
        'val': str(val_txt.resolve()),
        'test': 'test/images',
        'nc': NC,
        'names': CLASS_NAMES,
    }
    yaml_path = KFOLD_DIR / f'fold{i}_dataset.yaml'
    with open(yaml_path, 'w') as f:
        yaml.safe_dump(fold_yaml, f, sort_keys=False)
    fold_yaml_paths.append(yaml_path)

    print(f'fold {i}: {len(train_imgs)} train / {len(val_imgs)} val -> {yaml_path}')

fold 0: 283 train / 142 val -> /home/pritamthapa/projects/cervical_cancer/ml-models/models/organism_det/configs/kfold/fold0_dataset.yaml
fold 1: 283 train / 142 val -> /home/pritamthapa/projects/cervical_cancer/ml-models/models/organism_det/configs/kfold/fold1_dataset.yaml
fold 2: 284 train / 141 val -> /home/pritamthapa/projects/cervical_cancer/ml-models/models/organism_det/configs/kfold/fold2_dataset.yaml


### Backbone freeze warm-start callback

`freeze=10` freezes the first 10 backbone layers for the whole run by default. This callback fires
on `on_train_epoch_start` every epoch and, once at epoch 10, restores `requires_grad=True` on those
same layers so the rest of training proceeds unfrozen.

In [ ]:
UNFREEZE_EPOCH = 10
_unfreeze_state = {'done': False}

def unfreeze_backbone(trainer):
    """on_train_epoch_start callback: restore requires_grad on frozen backbone layers after warm-start.

    Uses >= plus a one-time flag (not ==) so this is also correct when resuming a fold from a
    checkpoint already past epoch 10 -- _setup_train re-freezes on every process start regardless of
    resume, and a bare '== 10' would never fire again once the resumed epoch counter has passed it,
    leaving the backbone permanently frozen for the rest of that run.

    No optimizer rebuild needed -- build_optimizer() groups ALL named_parameters() regardless of
    requires_grad (ultralytics/engine/trainer.py:1023-1034), so frozen params already sit inside the
    optimizer's param groups; PyTorch just skips params with grad=None until requires_grad flips back on.
    """
    if _unfreeze_state['done'] or trainer.epoch < UNFREEZE_EPOCH:
        return
    unfrozen = 0
    for name, param in trainer.model.named_parameters():
        if any(x in name for x in trainer.freeze_layer_names) and '.dfl' not in name:
            param.requires_grad = True
            unfrozen += 1
    _unfreeze_state['done'] = True
    print(f'[unfreeze_backbone] epoch {trainer.epoch}: restored requires_grad on {unfrozen} backbone params')

### Train YOLO26n across the 3 folds with the v3 recipe

Fresh `YOLO('yolo26n.pt')` per fold (reloads COCO-pretrained weights each time -- no leakage across
folds). Evaluates each fold's `best.pt` on its own held-out val partition and on the untouched test
set. ~30-35 min/fold expected (n-scale, 1280px, batch 8) -> ~1.5-1.75hr for all 3 folds.

In [ ]:
from ultralytics import YOLO

fold_results = []

for i in range(K):
    fold_yaml = fold_yaml_paths[i]
    run_name = f'fold{i}'

    model = YOLO('yolo26n.pt')
    model.add_callback('on_train_epoch_start', unfreeze_backbone)

    model.train(
        data=str(fold_yaml),
        project=str(CHECKPOINT_DIR),
        name=run_name,
        exist_ok=True,
        optimizer='auto',
        plots=False,
        save=True,
        verbose=True,
        **RECIPE,
    )

    best_path = CHECKPOINT_DIR / run_name / 'weights' / 'best.pt'
    best = YOLO(str(best_path))

    for split in ['val', 'test']:
        r = best.val(data=str(fold_yaml), split=split, imgsz=IMGSZ, verbose=False)
        row = {
            'fold': i, 'split': split,
            'mAP50': float(r.box.map50), 'mAP50_95': float(r.box.map),
            'precision': float(r.box.mp), 'recall': float(r.box.mr),
        }
        for c, name in enumerate(CLASS_NAMES):
            row[f'AP50_{name}'] = float(r.box.ap50[c]) if c < len(r.box.ap50) else None
        fold_results.append(row)
        print(f"fold {i} [{split:4s}]  mAP50={row['mAP50']:.4f}  P={row['precision']:.4f}  R={row['recall']:.4f}")

New https://pypi.org/project/ultralytics/8.4.90 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.60 🚀 Python-3.10.12 torch-2.12.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4090, 22554MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.35, compile=False, conf=None, copy_paste=0.6, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/home/pritamthapa/projects/cervical_cancer/ml-models/models/organism_det/configs/kfold/fold0_dataset.yaml, degrees=15.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=10, half=False, hsv_h=0.05, hsv_s=0.3, hsv_v=0.1, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.001, mask_ratio=4, max_det=300, mi

### Aggregate results across folds

Mean ± std per class across the 3 folds' val evaluations and the 3 folds' test evaluations --
this replaces a single noisy ~47-image val estimate with 3 independent ones. Compare against the
existing single-split v2 numbers in `training_report.md` §9 (test AP50: TV 0.443, BV 0.727,
Candida 0.381, Actinomyces 0.995).

In [ ]:
results_df = pd.DataFrame(fold_results)
out_path = METRICS_DIR / 'kfold_v3_recipe_validation.csv'
results_df.to_csv(out_path, index=False)
print(f'Saved per-fold results -> {out_path}\n')
print(results_df.to_string(index=False))

metric_cols = ['mAP50', 'mAP50_95', 'precision', 'recall'] + [f'AP50_{n}' for n in CLASS_NAMES]
print('\n=== Aggregated across folds (mean ± std) ===')
for split in ['val', 'test']:
    sub = results_df[results_df.split == split]
    print(f'\n-- {split} --')
    for col in metric_cols:
        print(f'  {col:<40} {sub[col].mean():.4f} ± {sub[col].std():.4f}')

### Reading the result / next step

Check, per class, before trusting this recipe further:
- **Actinomyces** precision/recall shouldn't have collapsed relative to v2's test AP 0.995 — that
  would mean `cls_pw=0.35` + `freeze=10` reignited the overfit that `copy_paste=0.6` fixed.
- **TV** should show some improvement over v2's test AP 0.443 now that `multi_scale` is capped at
  0.25 instead of 0.5.
- Fold-to-fold std tells you how much to trust any single number — a std comparable to the mean
  means the recipe's effect is not yet distinguishable from split noise at this sample size.

**Not done here (separate follow-up session):** train the actual YOLO26s deployment checkpoint on
the full pooled 425-image train+val set with whichever settings this validation confirms, then one
final honest evaluation on the untouched test set — mirroring notebook 06's cell 9 pattern.